# Example notebook Pydantic AI

Fill in the environment variables from the example env file (API KEY will be provided) and run the code below for an example how to use Agents from Pydantic AI

## Setup

In [ ]:
import os
from datetime import datetime

import mlflow
import pytz
from dotenv import load_dotenv
from pydantic_ai import Agent
from pydantic_ai.capabilities import Thinking, WebSearch
from pydantic_ai.models.openai import OpenAIResponsesModelSettings

load_dotenv()

# If mlflow server is not running, see README for more info
mlflow.set_tracking_uri("http://localhost:5068")
mlflow.set_experiment(f"agentic-ai-hackathon-{os.getenv('USER')}")

mlflow.pydantic_ai.autolog()

deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT", "")

## Init agent and settings

In [ ]:
settings = OpenAIResponsesModelSettings(
    openai_reasoning_effort="low",
    openai_reasoning_summary="detailed",
)

# Capabilities can be added to the agent, such as Thinking and WebSearch. You can also add your own custom tools (see example below).
agent = Agent(
    f"azure:{deployment}",
    instructions="Be concise, reply with one sentence.",
    capabilities=[Thinking(), WebSearch(local="duckduckgo")],
    model_settings=settings,
)

## Send request to agent

In [ ]:
result = await agent.run("What was the mass of the largest meteorite found in 2026?")
print(result.output)

In [ ]:
# Use all_messages to get the full conversation history structured as classes
result.all_messages()

## Use custom tool

In [ ]:
@agent.tool_plain
def get_current_time(timezone: str = "CET") -> str:
    """Get the current time as a string."""

    tz = pytz.timezone(timezone)
    return datetime.now(tz).isoformat()

In [ ]:
result = await agent.run("What is the current time in Tokyo?")
print(result.output)

In [ ]:
result.all_messages()